# OMNI MOVIE STUDIO — FREE GPU BRIDGE (Colab T4)
Full free stack: ComfyUI (Wan 2.2 animation) + Wav2Lip (lip-sync) + XTTS (Hindi voices).
Runtime -> Run all. At the end you get two public proxy URLs from Colab itself (free, no signup, no external tool) — paste them to your agent and it drives this GPU remotely.

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
import os
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/omni-movie'
os.makedirs(OUT, exist_ok=True)
print('drive ready:', OUT)

In [ ]:
# 1) ComfyUI + Wan 2.2 (free open weights) on 0.0.0.0
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI > /dev/null
%pip -q install -r /content/ComfyUI/requirements.txt
%cd /content/ComfyUI
import subprocess, threading, time, urllib.request
threading.Thread(target=lambda: subprocess.run(['python','main.py','--port','8188','--listen','0.0.0.0','--dont-print-server']), daemon=True).start()
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats'); break
    except Exception: time.sleep(2)
print('comfyui up')

In [ ]:
# 2) Wav2Lip — free lip-sync
!git clone --depth 1 https://github.com/Rudrabha/Wav2Lip /content/Wav2Lip > /dev/null
%cd /content/Wav2Lip
!mkdir -p checkpoints face_detection/detection/sfd
!wget -q https://github.com/justinjohn0306/Wav2Lip/releases/download/Models/wav2lip_gan.pth -O checkpoints/wav2lip_gan.pth
!wget -q https://github.com/justinjohn0306/Wav2Lip/releases/download/Models/s3fd.pth -O face_detection/detection/sfd/s3fd.pth
%pip -q install librosa==0.10.1 numba==0.58.1
print('wav2lip ready — env: LIPSYNC_ENGINE=wav2lip, WAV2LIP_SCRIPT=/content/Wav2Lip/inference.py')

In [ ]:
# 3) XTTS v2 — expressive Hindi TTS server (port 8020)
%pip -q install TTS==0.22.0 > /dev/null
import threading, subprocess
threading.Thread(target=lambda: subprocess.run(['python','/content/xtts_server.py']), daemon=True).start()
server = '''
from TTS.api import TTS
from http.server import BaseHTTPRequestHandler, HTTPServer
import io, json
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2')
class H(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == '/health': self.send_response(200); self.end_headers()
    def do_POST(self):
        n = int(self.headers.get('content-length', 0)); body = json.loads(self.rfile.read(n))
        wav = io.BytesIO()
        tts.tts_to_file(text=body['text'], language=body.get('language','hi'),
                        speaker_wav=body.get('speaker_wav'), file_path=body.get('out','/tmp/x.wav'))
        self.send_response(200); self.send_header('content-type','audio/wav'); self.end_headers()
        self.wfile.write(open(body.get('out','/tmp/x.wav'),'rb').read())
HTTPServer(('0.0.0.0', 8020), H).serve_forever()
'''
open('/content/xtts_server.py','w').write(server)
import threading
threading.Thread(target=lambda: __import__('subprocess').run(['python','/content/xtts_server.py']), daemon=True).start()
print('xtts booting on :8020')

In [ ]:
# 4) FREE public URLs — Colab's own kernel proxy (no account, no cost)
import time
time.sleep(60)  # let XTTS finish model download
from google.colab.output import eval_js
comfy_url = eval_js('google.colab.kernel.proxyPort(8188)')
xtts_url = eval_js('google.colab.kernel.proxyPort(8020)')
print('='*60)
print('PASTE THESE TO YOUR AGENT (keep private — anyone with the URL can use this GPU):')
print(f'COMFYUI_URL={comfy_url}')
print(f'XTTS_SERVER_URL={xtts_url}')
print('Wav2Lip runs inside this notebook; the agent renders shots via ComfyUI above.')
print('Disconnect the notebook when done to free the GPU.')
print('='*60)